# Visualizations for topological pre-training experiments

This notebook is for visualising the benchmarking results of different molecular featurisation approaches on QSAR tasks.

Contains statistcal tests and visualisations of benchmark results.

## Preamble

In [ ]:
run = {
    "vocab": True,
    "dist_plot": True,
    "sim_plot": True,
    "violin_plot": True,
}
save_figs = False

In [ ]:
# imports
# standard libraries
import glob
import logging
import os
import random
import re
import sys
import time
from itertools import combinations
from pathlib import Path
from string import ascii_lowercase

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

from topological_pretraining.data.mol import MorganGenerator
from topological_pretraining.data import load_dataset
from IPython.core.interactiveshell import InteractiveShell
from matplotlib.collections import PolyCollection
from matplotlib.patches import FancyBboxPatch, Patch, PathPatch
from pingouin import compute_effsize
from scikit_posthocs import posthoc_tukey_hsd, posthoc_wilcoxon
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    matthews_corrcoef,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from tqdm import tqdm

In [ ]:
# paths to directories
# for custom paths, set environment variables in an .env file
dotenv.load_dotenv(override=True)

DATA_DIR = os.environ.get("DATA_DIR", False)
if DATA_DIR:
    DATA_DIR = Path(DATA_DIR)
    if not DATA_DIR.exists():
        raise ValueError(f"DATA_DIR {DATA_DIR} does not exist.")
else:
    DATA_DIR = Path("../data")
    if not DATA_DIR.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = os.environ.get("RESULTS_DIR", "../results")
RESULTS_DIR = Path(RESULTS_DIR)
if not RESULTS_DIR.exists():
    raise ValueError(f"RESULTS_DIR {RESULTS_DIR} does not exist.")

HYPERPARAMS_DIR = os.environ.get("HYPERPARAMS_DIR", "../data/hyperparameters")
HYPERPARAMS_DIR = Path(HYPERPARAMS_DIR)
if not HYPERPARAMS_DIR.exists():
    raise ValueError(f"HYPERPARAMS_DIR {HYPERPARAMS_DIR} does not exist.")

FIGURES_DIR = os.environ.get("FIGURES_DIR", "./figures")
FIGURES_DIR = Path(FIGURES_DIR)
if not FIGURES_DIR.exists():
    raise ValueError(f"FIGURES_DIR {FIGURES_DIR} does not exist.")

(FIGURES_DIR / "biogen").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "molnet").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "chembl").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "muv").mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "tox21").mkdir(parents=True, exist_ok=True)

Path("./temp").mkdir(parents=True, exist_ok=True)

MODEL_DIR = os.environ.get("MODELS_DIR", "../pt_models")
MODEL_DIR = Path(MODEL_DIR)

LOGGING_LEVEL = int(os.environ.get("LOGGING_LEVEL", 20))
LOW_RES = bool(int(os.environ.get("LOW_RES", 1)))
IMG_SIZE_SCALE = float(os.environ.get("IMG_SIZE_SCALE", 1.0))
FILE_FORMAT = os.environ.get("FILE_FORMAT", "pdf")

In [ ]:
def is_interactive():
    """
    True if running in an interactive Jupyter frontend (VS Code, JupyterLab, classic),
    False if running headless via nbconvert.
    """
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if not ip:
            return False

        if ip.__class__.__name__ != "ZMQInteractiveShell":
            return True

        # Try to locate the connection file
        connection_file = getattr(ip, 'connection_file', None)
        if not connection_file:
            # Sometimes passed as "-f /path/to/connection.json"
            for idx, arg in enumerate(sys.argv):
                if arg == "-f" and idx + 1 < len(sys.argv):
                    connection_file = sys.argv[idx + 1]
                    break
                if arg.endswith(".json") and os.path.isfile(arg):
                    connection_file = arg
                    break

        if not connection_file or not os.path.isfile(connection_file):
            return True

        # Check how recently the file was created/modified
        mtime = os.path.getmtime(connection_file)
        age_seconds = time.time() - mtime

        if age_seconds < 5:
            return False

        return True
    except Exception:
        return False

In [ ]:
if not is_interactive():
    Path("./temp/logs").mkdir(parents=True, exist_ok=True)
    all_logs = glob.glob("temp/logs/*.log")
    if len(all_logs) == 0:
        logger_path = "temp/logs/tmp0.log"
    else:
        numbers = [int(re.findall(r"\d+", l)[0]) for l in all_logs]
        last = max(numbers)
        logger_path = f'temp/logs/tmp{last + 1}.log'
    logging.basicConfig(
        level=LOGGING_LEVEL, 
        handlers=[
            logging.FileHandler(filename=logger_path, mode="w+"),
            logging.StreamHandler(sys.__stderr__),
            logging.StreamHandler(sys.__stdout__),
        ],
        format="[%(asctime)s] %(levelname)s: %(message)s", 
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    logger = logging.getLogger(name="my_logger")
    logger.propagate = True
    logger.info("Logging started")
    logger.info(f"Logger path: {logger_path}")
else:
    logger_path = None
    logging.basicConfig(
        level=logging.INFO, 
        handlers=[
            logging.StreamHandler(sys.__stderr__),
            logging.StreamHandler(sys.__stdout__),
        ],
        format="[%(asctime)s] %(levelname)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    logger = logging.getLogger(name="my_logger")
    logger.info("Logging started")

In [ ]:
logger.info(is_interactive())

In [ ]:
logger.info(f"Data path: {DATA_DIR}")
logger.info(f"Results path: {RESULTS_DIR}")
logger.info(f"Hyperparameters path: {HYPERPARAMS_DIR}")
logger.info(f"Figures path: {FIGURES_DIR}")
logger.info(f"Model path: {MODEL_DIR}")
logger.info(f"Low resolution: {LOW_RES}")

In [ ]:
# Exception Hook for Jupyter 
def notebook_exception_handler(shell, etype, evalue, tb, tb_offset=None):
    """
    This will run for uncaught exceptions in Jupyter cells.
    """
    logger.critical("HOOK TRIGGERED:", etype, evalue)
    logger.critical("Uncaught exception", exc_info=(etype, evalue, tb))

# Attach the handler to catch *all* exceptions
InteractiveShell.instance().set_custom_exc((Exception,), notebook_exception_handler)

In [ ]:
# seaborn settings
logger.info("Setting seaborn settings")
sns.set(
    context="notebook",
    palette="deep",
)
sns.set_style("darkgrid", {"grid.color": ".6", "grid.linestyle": ":"})

In [ ]:
# dict of benchmark datasets and display names
benchmark_dataset_names = {
    # Biogen
    "Efflux": "Efflux", "Human_CLint": "Human CLint", "Human_PPB": "Human PPB",
    "Rat_CLint": "Rat CLint", "Rat_PPB": "Rat PPB", "Solu": "Solubility",
    # ChEMBL affinity
    "DRD2":"DRD2", "FactorXA":"Factor $\\mathrm{X_A}$",
    # MoleculeNet
    "ESOL":"ESOL", "ESOL_restricted": "ESOL Drug-Like Only", "FreeSolv":"FreeSolv", "Lipo":"Lipophilicity",
    # MoleculeNet - MUV
    "MUV466":"MUV466", "MUV548":"MUV548", "MUV600":"MUV600",
    "MUV644":"MUV644", "MUV652":"MUV652", "MUV689":"MUV689",
    "MUV692":"MUV692", "MUV712":"MUV712", "MUV713":"MUV713",
    "MUV733":"MUV733", "MUV737":"MUV737", "MUV810":"MUV810",
    "MUV832":"MUV832", "MUV846":"MUV846", "MUV852":"MUV852",
    "MUV858":"MUV858", "MUV859":"MUV859",
    # Tox21
    "NR_AR":"NR AR", "NR_AR_LBD":"NR AR LBD", "NR_AhR":"NR AhR",
    "NR_Aromatase":"NR Aromatase", "NR_ER":"NR ER", "NR_ER_LBD":"NR ER LBD",
    "NR_PPAR_gamma":"NR PPAR $\\gamma$", "SR_ARE":"SR ARE", "SR_ATAD5":"SR ATAD5",
    "SR_HSE":"SR HSE", "SR_MMP":"SR MMP", "SR_p53":"SR p53"
}

In [ ]:
# subsets of datasets
biogen_datasets = ["Human_PPB", "Rat_PPB", "Human_CLint", "Rat_CLint", "Efflux", "Solu"]
chembl_datasets = ["DRD2", "FactorXA"]
molnet_regression_datasets = ["ESOL", "FreeSolv", "Lipo"]
esol_datasets = ["ESOL", "ESOL_restricted"]
muv_datasets = [i for i in benchmark_dataset_names if "MUV" in i]
tox21_datasets = [i for i in benchmark_dataset_names if (i[:2] == "NR") or (i[:2] == "SR")]

In [ ]:
# load base standardizer settings
with open("../config/base/standardizer.yaml") as f:
    standardizer_kwargs = yaml.safe_load(f)["standardizer"]
    standardizer_kwargs["n_jobs"] = -1

In [ ]:
# loading datasets
logger.info("Loading datasets")

# Load QMugs dataset
logger.info("Loading QMugs dataset")
qmugs_df = load_dataset("QMugs", DATA_DIR, verbose=True, standardizer=standardizer_kwargs)

benchmark_datasets = {}
for i in benchmark_dataset_names:
    if i == "ESOL_restricted": continue
    logger.info(f"Loading dataset {i}")
    benchmark_datasets[i] = load_dataset(i, DATA_DIR, verbose=True, standardizer=standardizer_kwargs)


In [ ]:
# Enrichment score function for VS (MUV datasets)
logger.info("Defining enrichment score function")
def enrichment_score(y_true, y_pred, percentile: float = 0.05, ):
    if len(np.unique(y_true)) == 1:
        Warning(f"y_true has only one unique value: {np.unique(y_true)}")
        return np.nan
    n_total = len(y_true)
    a_total = sum(y_true)
    fa_total = a_total / n_total

    idx = np.argsort(y_pred)[::-1]
    n_top = int(n_total * percentile)
    a_top = sum(y_true[idx][:n_top])
    fa_top = a_top / n_top
    return fa_top / fa_total

In [ ]:
# dictionary for metric functions
logger.info("Defining metric functions")
metrics_dict = {
    "classification": {
        "Accuracy": lambda y_true, y_pred: accuracy_score(y_true, (y_pred > 0.5).astype(int)),
        "AUCPR": average_precision_score,
        "AUROC": roc_auc_score,
        "F1": lambda y_true, y_pred: f1_score(y_true, (y_pred > 0.5).astype(int)),
        "MCC": lambda y_true, y_pred: matthews_corrcoef(y_true, (y_pred > 0.5).astype(int)),
        "Precision": lambda y_true, y_pred: precision_score(y_true, (y_pred > 0.5).astype(int)),
        "Recall": lambda y_true, y_pred: recall_score(y_true, (y_pred > 0.5).astype(int)),
    },
    "regression": {
        "MAE": mean_absolute_error,
        "R2": r2_score,
        "Pearson": lambda y_true, y_pred: stats.pearsonr(y_true, y_pred)[0],
        "Spearman": lambda y_true, y_pred: stats.spearmanr(y_true, y_pred)[0],
        "MSE": mean_squared_error,
        "MAPE": mean_absolute_percentage_error,
    },
    "virtual_screening": {
        "EF05": lambda y_true, y_pred: enrichment_score(y_true, y_pred, percentile=0.005),
        "EF1": lambda y_true, y_pred: enrichment_score(y_true, y_pred, percentile=0.01),
        "EF5": lambda y_true, y_pred: enrichment_score(y_true, y_pred, percentile=0.05),
        "EF10": lambda y_true, y_pred: enrichment_score(y_true, y_pred, percentile=0.1),
    }
}
metrics_dict["regression"]["RMSE"] = lambda y_true, y_pred: np.sqrt(metrics_dict["regression"]["MSE"](y_true, y_pred))
metrics_dict["virtual_screening"].update(metrics_dict["classification"])

In [ ]:
# metric names for plotting
metric_display_names = {
    "R2": "$\\uparrow\\mathrm{R^2}$",
    "Pearson": "$\\uparrow\\rho$",
    "MAPE": "$\\downarrow\\mathrm{MAPE} (\\%)$",
    "AUROC": "$\\uparrow\\mathrm{AUROC}$",
    "AUCPR": "$\\uparrow\\mathrm{AUCPR}$",
    "MCC": "$\\uparrow\\mathrm{MCC}$",
    "EF10": "$\\uparrow\\mathrm{EF_{10\\%}}$",
    "EF5": "$\\uparrow\\mathrm{EF_{5\\%}}$",
    "EF1": "$\\uparrow\\mathrm{EF_{1\\%}}$",
    "EF05": "$\\uparrow\\mathrm{EF_{0.5\\%}}$",
}

In [ ]:
# calculate the maximum possible enrichment factor
def get_ef_max(n, a, frac):
    fa_total = a/n
    fa_x = a / (n*frac)
    return fa_x/fa_total

In [ ]:
# metric range limits
metric_range_limits = {
    "R2": (-np.inf, 1),
    "Pearson": (-1, 1),
    "MAPE": (0, np.inf),
    "AUROC": (0, 1),
    "AUCPR": (0, 1),
    "MCC": (-1, 1),
    "EF10": (0, get_ef_max(15000, 15, 0.1)), # 15000 molecules and 15 actives in each MUV dataset
    "EF5": (0, get_ef_max(15000, 15, 0.05)),
    "EF1": (0, get_ef_max(15000, 15, 0.01)),
    "EF05": (0, get_ef_max(15000, 15, 0.005)),
}

In [ ]:
# sorting ascending or descending
metric_sort_reverse = {
    "MAE": False,
    "AUCPR": True,
    "R2": True,
    "Pearson": True,
    "MAPE":False,
    "AUROC": True,
    "MCC": True,
    "EF10": True,
    "EF5": True,
    "EF1": True,
    "EF05": True,
}

In [ ]:
# method display names
method_display_names = {
    "pt_gin": "PT-GIN",
    "scratch_gin": "Scratch GIN",
    "sns": "ECFP$_{\\mathrm{S&S}}$",
    "ecfp": "ECFP$_{\\mathrm{hashed}}$",
    "fcfp": "FCFP$_{\\mathrm{hashed}}$",
}

method_display_names = {
    "pt_gin": "$\\text{PT-GIN}$",
    "scratch_gin": "$\\text{Scratch GIN}$",
    "sns": "$\\text{ECFP}_{\\mathrm{S&S}}$",
    "ecfp": "$\\text{ECFP}_{\\mathrm{hashed}}$",
    "fcfp": "$\\text{FCFP}_{\\mathrm{hashed}}$",
}
method_save_names = {
    "pt_gin": "PT_GIN",
    "scratch_gin": "Scratch_GIN",
    "sns": "SNS",
    "ecfp": "ECFP",
    "fcfp": "FCFP",
}
def get_method_name(substring, display_or_save = "d", tanimoto=False):
    if tanimoto:
        if "tanimoto" not in substring: return "0.5"
        else: return f"{substring[-2]}.{substring[-1]}"

    name = [i for i in method_display_names if substring.startswith(i)]
    if len(name) == 0:
        raise ValueError(f"Method name {substring} not found in method_display_names.")
    if len(name) > 1:
        raise ValueError(f"Method name {substring} is ambiguous, found multiple matches: {name}")
    substring = name[0]
    if display_or_save == "d":
        return method_display_names[substring]
    elif display_or_save == "s":
        return method_save_names[substring]
    else:
        raise ValueError("display_or_save must be 'd' or 's'")

## General Plotting Functions

In [ ]:
# renfer box around an axis
def render_box(
    fig, axes, 
    boxstyle="round,pad=0.01",
    edgecolor='grey',
    linestyle='dotted',          # dotted border
    linewidth=float(2*IMG_SIZE_SCALE),
    facecolor='none',            # transparent fill
    zorder=10,
    **kwargs
):
    positions = [ax.get_position() for ax in axes]
    renderer = fig.canvas.get_renderer()
    bboxes = [ax.get_tightbbox(renderer) for ax in axes]

    # Convert bboxes from display (pixel) to figure coordinates
    positions = [bbox.transformed(fig.transFigure.inverted()) for bbox in bboxes]
    x0 = min(pos.x0 for pos in positions)
    y0 = min(pos.y0 for pos in positions)
    x1 = max(pos.x1 for pos in positions)
    y1 = max(pos.y1 for pos in positions)
    # Create a FancyBboxPatch around the subplot
    box = FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0,       # width and height
        boxstyle=boxstyle,   # rounded box
        edgecolor=edgecolor,
        linestyle=linestyle,          # dotted border
        linewidth=linewidth,
        facecolor=facecolor,            # transparent fill
        transform=fig.transFigure,   # use figure coords
        zorder=zorder,
        **kwargs,
    )
    # Add the patch to the figure
    fig.patches.append(box)

In [ ]:
if LOW_RES:
    dpi = 50
else:
    dpi = 300
logger.info(f"Setting seaborn savefig dpi to {dpi}")
sns.set_theme(
    rc={
        "savefig.dpi": dpi,
        "savefig.format": FILE_FORMAT,
    }
)

## Substructure Vocabularies

In [ ]:
runner = run["vocab"]

### QMugs: Unique Molecules vs Substructure Counts

In [ ]:
# Morgan substructure environments for QMugs
if runner:
    logger.info("Generating Morgan substructure environments for QMugs")
    qmugs_mols = qmugs_df.rdkit_mols
    generator = MorganGenerator(radius=2, chirality=True,)
    qmugs_envs = []
    with tqdm(total=len(qmugs_mols), desc="Generating substructure environments") as pbar:
        counter = 0
        for mol in qmugs_mols:
            if mol is None:
                qmugs_envs.append(None)
                pbar.update(1)
                continue
            env = generator.environments(mol)
            qmugs_envs.append(env)
            if float(pbar.n) / float(pbar.total)*100 > counter:
                counter += 1
                logger.info(": " + str(pbar))

            pbar.update(1)
        logger.info(": " + str(pbar))

In [ ]:
# subset sustructure counts
"""
iterate over increasing subset sizes
and calculate the number of unique substructures
and the ratio of unique substructures to the number of molecules
in the subset
"""
if runner:
    logger.info("Calculating subset structure counts")
    number_of_molecules = []
    number_of_substructures = []
    ratio_substructures_molecules = []
    i = 10
    qmugs_envs_without_none = [i for i in qmugs_envs if i is not None]
    while 2**i < len(qmugs_envs):
        subset_size = 2**i
        number_of_molecules.append(subset_size)
        subset_of_molcules = random.sample(
            qmugs_envs_without_none, subset_size
        )
        subset_of_molcules = np.vstack(subset_of_molcules)
        substructures = len(np.unique(subset_of_molcules))
        number_of_substructures.append(substructures)
        ratio_substructures_molecules.append(substructures / subset_size)
        i += 1

In [ ]:
# plot the number of substructures as a function of the number of molecules
if runner:
    logger.info("Plotting number of substructures as a function of the number of molecules")
    sns.set_theme(
        rc={
            'figure.figsize':(int(12*IMG_SIZE_SCALE),int(16*IMG_SIZE_SCALE)),
            "axes.labelsize": int(18*IMG_SIZE_SCALE),
            "xtick.labelsize": int(16*IMG_SIZE_SCALE),
            "xtick.major.size": int(5*IMG_SIZE_SCALE),
            "ytick.labelsize": int(16*IMG_SIZE_SCALE),
            "ytick.major.size": int(5*IMG_SIZE_SCALE),
            "axes.titlesize":int(20*IMG_SIZE_SCALE),
            "figure.subplot.hspace": 0.65,
            "figure.subplot.wspace": 0.3,
            "legend.title_fontsize": int(20*IMG_SIZE_SCALE),
        }
    )
    fig, axes = plt.subplots(2,1)
    axes = axes.ravel()
    ax1 = axes[0]
    ylabel1 = "$\\frac{\\text{Number of  Unique Substructures}}{\\text{Number of Molecules}}$"
    ylabel2 = "Number of Unique Substructures"

    ax2 = ax1.twinx()
    ax1=sns.lineplot(
        x=number_of_molecules,
        y=ratio_substructures_molecules,
        ax=ax1, color="black", linestyle="--",
        label=ylabel1,
        legend=False,
        linewidth=0.5,
    )
    ax2=sns.lineplot(
        x=number_of_molecules,
        y=number_of_substructures,
        ax=ax2,
        color="blue", 
        label=ylabel2,
        legend=False,
        linewidth=0.5,
    )
    ax1.tick_params(axis='both', which='major', labelsize=int(14*IMG_SIZE_SCALE))
    ax2.tick_params(axis='both', which='major', labelsize=int(14*IMG_SIZE_SCALE))
    ax1.set_yticks(
        np.arange(0, max(ratio_substructures_molecules)+2, 2), 
    )
    ax1.set_ylim(0, 12.5)
    ax1.set_xlabel("Number of molecules", fontsize=int(14*IMG_SIZE_SCALE))
    ax1.set_ylabel(ylabel1, fontsize=int(18*IMG_SIZE_SCALE))
    ax1.tick_params(left=False, bottom=False)
    ax2.set_ylim(0, 312500)
    ax2.set_ylabel(ylabel2, fontsize=int(14*IMG_SIZE_SCALE))
    ax2.tick_params(right=False, bottom=False)

    handles, labels = ax1.get_legend_handles_labels()
    handles.extend(ax2.get_legend_handles_labels()[0])
    labels.extend(ax2.get_legend_handles_labels()[1])
    ax1.set_title(f"({ascii_lowercase[0]})", loc="left", pad=int(20*IMG_SIZE_SCALE))

    ax1.legend(
        handles=handles,
        labels=labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.2),
        prop={"size":int(16*IMG_SIZE_SCALE)},
        ncols=2,
    )
    render_box(fig, [ax1, ax2])

### QMugs: substructure overlap with benchmark datasets

In [ ]:
# get substructure environments for each tanimoto filter split
if runner:
    logger.info("Getting substructure environments for each tanimoto filter split")
    tanimoto_splits = [(qmugs_df[qmugs_df[i] == 1].index, float(i.split('_')[-1])) for i in qmugs_df.columns if 'tanimoto_filter' in i]
    qmugs_split_envs = {}
    for split_indexes, threshold in tanimoto_splits:
        qmugs_split_envs[threshold] = [qmugs_envs[j] for j in split_indexes]

In [ ]:
# get substructure environments for each benchmark dataset
if runner:
    logger.info("Getting substructure environments for each benchmark dataset")
    parent_dir = DATA_DIR / "temp"
    parent_dir.mkdir(parents=True, exist_ok=True)
    if (parent_dir / "substructure_environments.pt").exists():
        benchmark_substructure_environments = torch.load(
            parent_dir / "substructure_environments.pt",
            map_location="cpu",
            weights_only=False,
        )
    else:
        benchmark_substructure_environments = {}
        for dataset_name, dataset in benchmark_datasets.items():
            benchmark_substructure_environments[dataset_name] = []
            benchmark_mols = dataset.rdkit_mols
            with tqdm(total=len(benchmark_mols), desc=f"Generating substructure environments for {dataset_name}") as pbar:
                counter = 0
                for mol in benchmark_mols:
                    if mol is None:
                        if float(pbar.n) / float(pbar.total)*100 > counter:
                            counter += 1
                            logger.info(": " + str(pbar))
                        pbar.update(1)
                        continue
                    env = generator.environments(mol)
                    benchmark_substructure_environments[dataset_name].append(env)
                    if float(pbar.n) / float(pbar.total)*100 > counter:
                        counter += 1
                        logger.info(": " + str(pbar))
                    pbar.update(1)
                logger.info(": " + str(pbar))
                
        torch.save(
            benchmark_substructure_environments,
            parent_dir / "substructure_environments.pt",
        )

In [ ]:
# calculate % coverage of benchmark substructure environments in QMugs
if runner:
    logger.info("Calculating % coverage of benchmark substructure environments in QMugs")
    parent_dir = DATA_DIR / "temp"
    parent_dir.mkdir(parents=True, exist_ok=True)
    if (parent_dir / "qmugs_coverage.csv").exists():
        df = pd.read_csv(parent_dir / "qmugs_coverage.csv")
    else:
        df = {
            "Tanimoto Threshold": [],
            "Dataset": [],
            "Coverage (%)": [],
            "Name": [],
        }
        with tqdm(total=len(qmugs_split_envs), desc="Calculating coverage") as pbar:
            for bmark in benchmark_substructure_environments:
                for thr in qmugs_split_envs:
                    qmugs_set = set(np.concatenate(qmugs_split_envs[thr]).ravel().astype(int).tolist())
                    benchmark_set = set(np.concatenate(benchmark_substructure_environments[bmark]).ravel().astype(int).tolist())
                    coverage = len(benchmark_set.intersection(qmugs_set)) / len(benchmark_set) * 100
                    df["Coverage (%)"].append(coverage)
                    df["Tanimoto Threshold"].append(thr)
                    df["Dataset"].append(bmark.replace("_", " "))
                    df["Name"].append(bmark)

                logger.info(": " + str(pbar))
                pbar.update(1)
            logger.info(": " + str(pbar))
        df = pd.DataFrame(df)
        df.to_csv(parent_dir / "qmugs_coverage.csv", index=False)

In [ ]:
# plot coverage of benchmark datasets
if runner:
    df.Dataset = df.Dataset.apply(lambda x: "Solubility" if x == "Solu" else x) # rename
    logger.info("Plotting coverage of benchmark datasets")

    ax = axes[1]

    partial_ax = sns.barplot(
        data=df[df["Name"].isin(biogen_datasets)],
        x="Tanimoto Threshold",
        y="Coverage (%) (2048)",
        hue="Dataset",
        alpha=0.7,
        palette="colorblind",
        legend=False,
        ax=ax,
        linewidth=0,
    )
    for i,thisbar in enumerate(partial_ax.patches):
        # Set a different hatch for each bar
        thisbar.set_hatch("//////")

    full_ax = sns.barplot(
        data=df[df["Name"].isin(biogen_datasets)],
        x="Tanimoto Threshold",
        y="Coverage (%) (Full)",
        hue="Dataset",
        alpha=0.5,
        palette="colorblind",
        legend=True,
        ax=ax,
        linewidth=0,
        edgecolor="black",
    )

    handles, labels = full_ax.get_legend_handles_labels()
    handles.extend([
        Patch(
            edgecolor="black",
            facecolor="white",
            label="All QMugs Substructures",
            linewidth=0.5,
        ),
        Patch(
            edgecolor="black",
            facecolor="white",
            label="2048 Most Frequent QMugs Substructures",
            hatch="//////",
            linewidth=0.5,
        )
    ])
    full_ax.get_legend().remove()
    ax.legend(
        handles=handles,
        loc="upper center",
        ncol=4,
        bbox_to_anchor=(0.5, -0.2),
        prop={'size': int(16*IMG_SIZE_SCALE)},
    )

    ax.set_ylim(0, 100)
    ax.set_ylabel("Substructure Coverage (%)", fontsize=int(18*IMG_SIZE_SCALE))
    ax.set_title(f"({ascii_lowercase[1]})", loc="left", pad=int(20*IMG_SIZE_SCALE))
    render_box(fig, [ax])
    plt.show()
    if save_figs:
        fig.savefig(
            FIGURES_DIR / "extended/qmugs_coverage",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    # plt.close()

## Load Results

In [ ]:
# flatten and nest dictionary functions
def flatten_dict(d, sep=","):
    """
    Flatten a nested dictionary.
    """
    flat_dict = {}
    for key, value in d.items():
        if isinstance(value, dict):
            for sub_key, sub_value in flatten_dict(value, sep=sep).items():
                flat_dict[f"{key}{sep}{sub_key}"] = sub_value
        else:
            flat_dict[key] = value
    return flat_dict

def renest_dict(d, sep=","):
    """
    Renest a flattened dictionary.
    """
    nested_dict = {}
    for key, value in d.items():
        parts = key.split(sep)
        current_level = nested_dict
        for part in parts[:-1]:
            if part not in current_level:
                current_level[part] = {}
            current_level = current_level[part]
        current_level[parts[-1]] = value
    return nested_dict


In [ ]:
# load metrics results from file
logger.info("Load metrics for benchmark datasets")
results_path = RESULTS_DIR / "benchmark_results.npz"
if results_path.exists():
    results = np.load(results_path, allow_pickle=False,)
    results = renest_dict(results)
else:
    logger.warning(f"Results file {results_path} does not exist.")

In [ ]:
# calculate metrics if not already present
# no longer needed as all results are precomputed and stored in benchmark_results.npz
"""
if len(results) == 0:
    logger.info("Metrics results are empty, calculating metrics")
    with tqdm(total=len(benchmark_datasets), desc="Calculating metrics") as pbar:
        counter = 0
        for name, dataset in benchmark_datasets.items():
            if name in results:
                print(f"Results for {name} already present, skipping.")
                if float(pbar.n) / float(pbar.total)*100 > counter:
                    counter += 1
                    logger.info(": " + str(pbar))
                pbar.update(1)
                continue
            else:
                results[name] = {}
            search_path = str(RESULTS_DIR / "**" / f"{name.lower()}_preds.npz")
            pred_results_paths = glob.glob(search_path, recursive=True)
            if len(pred_results_paths) == 0:
                logger.warning(f"No prediction results found for {name}, skipping.")
                pbar.update(1)
                continue
            y_true = dataset.y.values
            metrics_to_calculate = metrics_dict[dataset.task]
            if "MUV" in name:
                metrics_to_calculate = metrics_dict["virtual_screening"]
            for pred_path in pred_results_paths:
                pred_path = Path(pred_path)
                method = pred_path.parent.stem
                if method not in results[name]:
                    results[name][method] = {}
                else:
                    continue
                preds = np.load(pred_path)["arr_0"]
                for i, (train, test) in enumerate(dataset.splits):
                    y_true_test = y_true[test]
                    y_pred_test = preds[i, test]
                    for metric_name, metric_func in metrics_to_calculate.items():
                        if metric_name == "MAPE":
                            num_zeros = np.sum(y_true_test == 0)
                            y_pred_test = y_pred_test[y_true_test != 0]
                            y_true_test = y_true_test[y_true_test != 0]

                        if metric_name not in results[name][method]:
                            results[name][method][metric_name] = np.full((dataset.num_splits), np.nan)
                        results[name][method][metric_name][i] = metric_func(y_true_test, y_pred_test)
            flattened_results = flatten_dict(results)
            np.savez_compressed(
                RESULTS_DIR / "benchmark_results.npz",
                **flattened_results,
                allow_pickle=False,
            )
            if float(pbar.n) / float(pbar.total)*100 > counter:
                counter += 1
                logger.info(": " + str(pbar))
            pbar.update(1)
        logger.info(": " + str(pbar))
"""

In [ ]:
# get ESOL restricted
# no longer needed as ESOL_restricted results are precomputed and stored in benchmark_results.npz

if "ESOL_restricted" not in results:
    esol_splits = list(benchmark_datasets["ESOL"].splits)
    y_true = benchmark_datasets["ESOL"].y.values
    y_true_druglike_idx = (y_true > 0) & (y_true < 3)

    esol_path = str(RESULTS_DIR / "**" / "esol_preds.npz")
    results["ESOL_restricted"] = {}
    for pred_file in glob.glob(esol_path, recursive=True):
        pred_file = Path(pred_file)
        method_name = pred_file.parent.name
        results["ESOL_restricted"][method_name] = {}
        preds = np.load(pred_file)["arr_0"]
        for i, (train, test) in enumerate(esol_splits):
            test_y = y_true[test]
            druglike_idx = (test_y > 0) & (test_y < 3)
            test_y_druglike = test_y[druglike_idx]
            pred_y = preds[i, test]
            pred_y = pred_y[druglike_idx]
            for metric, func in metrics_dict["regression"].items():
                out = results["ESOL_restricted"][method_name].get(metric, np.full(1000, np.nan))
                out[i] = func(test_y_druglike, pred_y)
                results["ESOL_restricted"][method_name][metric] = out

    flattened_results = flatten_dict(results)
    np.savez_compressed(
        RESULTS_DIR / "benchmark_results.npz",
        **flattened_results,
        allow_pickle=False,
    )


In [ ]:
# filter out test set samples with 0 true values for MAPE calculation
for dataset in results:
    if dataset == "ESOL_restricted":
        continue
    _task = benchmark_datasets[dataset].task
    _num_zeros = (benchmark_datasets[dataset].y == 0).sum()
    if _task == "regression" and _num_zeros > 0:
        logger.info(f"Dataset: {dataset}. Number of 0s: {_num_zeros}")
    else:
        continue
    all_splits = list(benchmark_datasets[dataset].splits)
    y_true = benchmark_datasets[dataset].y.values
    logger.info(f"Calculating MAPE values for {dataset} without 0 true values.")
    for method in results[dataset]:
        if "MAPE" in results[dataset][method]:
            pred_path =glob.glob(str(RESULTS_DIR / "**" / method / f"{dataset.lower()}_preds.npz"), recursive=True)
            assert len(pred_path) == 1, f"Multiple prediction files found for {dataset} and {method}. Pred_path: {pred_path}"
            pred_path = pred_path[0]
            preds = np.load(pred_path)["arr_0"]
            for i, (train, test) in enumerate(all_splits):
                y_true_test = y_true[test]
                y_pred_test = preds[i, test]
                y_pred_test = y_pred_test[y_true_test != 0]
                y_true_test = y_true_test[y_true_test != 0]
                _mape = metrics_dict["regression"]["MAPE"](y_true_test, y_pred_test)
                results[dataset][method]["MAPE"][i] = _mape
    flattened_results = flatten_dict(results)
    np.savez_compressed(
        RESULTS_DIR / "benchmark_results.npz",
        **flattened_results,
        allow_pickle=False,
    )


## Functions for Sorting Methods

In [ ]:
logger.info("Setting up functions for sorting methods")

In [ ]:
# define splits for tuning and plotting/testing
def get_tuning_splits():
    return np.arange(0, 5, dtype=int)

def get_test_splits(dataset, group_by_fold=True):
    splits_to_exclude = benchmark_datasets[dataset].splits_to_exclude_for_metrics
    if group_by_fold:
        indexes = [
            np.array([i for i in range(j*5,(j+1)*5) if i not in splits_to_exclude], dtype=int)
            for j in range(1, 200)
        ]
        return indexes
    else:
        indexes = np.array([i for i in range(5,1000) if i not in splits_to_exclude], dtype=int)
        return indexes

In [ ]:
# get array of metrics from results
def get_metrics(
    dataset, method, metric, splits = np.arange(0, 1000, dtype=int)
):
    try:
        if isinstance(splits, list):
            arr = np.array([results[dataset][method][metric][fold].mean() for fold in splits])
        else:
            arr = results[dataset][method][metric][splits]
    except Exception as e:
        print(f"Failed at:\n\tdataset = {dataset};\n\tmethod = {method};\n\tmetric = {metric}")
        raise ValueError(f"Error: {e}")
    if metric == "MAPE": arr *= 100
    return arr

In [ ]:
# get list of methods for particular dataset which match key
def find_methods(
    dataset, substring, 
    key = None,
):
    if key is None:
        key = lambda x, y: True if x.startswith(y) and ("filter" not in x) else False
    methods_list = list(results[dataset].keys())
    return [i for i in methods_list if key(i, substring)]

In [ ]:
# get method to use for plotting
def get_method_to_plot(
    dataset,
    substring,
    metric = None,
    **kwargs 
):
    if metric is None:
        if benchmark_datasets[dataset].task == "regression":
            metric = "MAE"
        else:
            metric = "AUCPR"
    methods_list = find_methods(dataset, substring, **kwargs)
    tuning_means = {i: get_metrics(dataset, i, metric, get_tuning_splits()).mean() for i in methods_list}
    tuning_means = dict(sorted(tuning_means.items(), key=lambda i: i[1], reverse=metric_sort_reverse[metric]))
    best_method = next(iter(tuning_means.keys()))
    return best_method
    

## Metric Distribution Plots

In [ ]:
runner = run["dist_plot"]
if runner:
    logger.info("Running metric distribution plots")
    default_img_size_scale = IMG_SIZE_SCALE
    IMG_SIZE_SCALE = 1.0

### Functions

In [ ]:
# prepare grid for normal distribution plots
def prep_grid(nrow=1):
    fig = plt.figure()
    height_ratios = [1 if i%3 == 0 else 5 if i%3==1 else 3 for i in range(nrow*3)]
    gs = fig.add_gridspec(3*nrow, 2, height_ratios=height_ratios, hspace=0.1, wspace=0.325)
    return fig, gs

In [ ]:
# plot distribution of array as box plot, histogram, and qq plots
def plot_distribution(array, fig, gs, row=0, label=None):
    ax_box = fig.add_subplot(gs[row*3, 0])
    ax_hist = fig.add_subplot(gs[row*3+1, 0], sharex=ax_box)
    ax_qq = fig.add_subplot(gs[row*3:row*3+2, 1])
    sns.boxplot(array, ax=ax_box, orient="h")
    if array.mean() == 0: 
        # edge case for MCC with all 0
        ax_hist.set_xlim(-1,1)
        sns.histplot(array, ax=ax_hist, bins=np.linspace(-1, 1, 20))
    else: sns.histplot(array, ax=ax_hist)
    stats.probplot(array, dist="norm", plot=ax_qq)
    

    ax_qq.set_title("")
    ax_box.set_title(f"({ascii_lowercase[row*2]})", loc="left", pad=int(20*IMG_SIZE_SCALE))
    ax_box.get_xaxis().set_visible(False)
    ax_qq.set_title(f"({ascii_lowercase[row*2+1]})", loc="left", pad=int(20*IMG_SIZE_SCALE))
    if label is not None:
        ax_qq.set_ylabel(label)
        ax_hist.set_xlabel(label)

    
    render_box(fig, [ax_box, ax_hist])
    render_box(fig, [ax_qq])
    return ax_box, ax_hist, ax_qq

### Regression Distributions

In [ ]:
# set sns theme

sns.set_theme(
    rc={
        "figure.figsize": (int(15*IMG_SIZE_SCALE), int(20*IMG_SIZE_SCALE)),
        "axes.labelsize": int(16*IMG_SIZE_SCALE),
        "xtick.labelsize": int(14*IMG_SIZE_SCALE),
        "ytick.labelsize": int(14*IMG_SIZE_SCALE),
        "axes.titlesize":int(20*IMG_SIZE_SCALE)
    }
)

In [ ]:
# biogen distributions
if runner:
    logger.info("Plotting Biogen distributions")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin", "scratch_gin", "ecfp", "sns", "fcfp"]
    break_loop = False
    for dataset in biogen_datasets:
        parent_path = FIGURES_DIR / f"normality/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                arr = get_metrics(dataset, method, metric, get_test_splits(dataset))
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)

            if save_figs:
                save_name = f"dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break

In [ ]:
# chembl distributions
if runner:
    logger.info("Plotting ChEMBL distributions")
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    break_loop = False
    for dataset in chembl_datasets:
        parent_path = FIGURES_DIR / f"normality/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                arr = get_metrics(dataset, method, metric, get_test_splits(dataset))
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)
            if save_figs:
                save_name = f"dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break

In [ ]:
# molnet regression distributions
if runner:
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    logger.info("Plotting MoleculeNet regression distributions")
    break_loop = False
    for dataset in molnet_regression_datasets:
        parent_path = FIGURES_DIR / f"normality/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                splits = get_test_splits(dataset)
                arr = get_metrics(dataset, method, metric, splits)
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)
            if save_figs:
                save_name = f"dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break

### Classification Distributions

In [ ]:
# MUV classification distributions AUROC AUCPR MCC
if runner:
    logger.info("Plotting MUV classification distributions")
    metrics = ["AUROC", "AUCPR", "MCC"]
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    break_loop = False
    for dataset in muv_datasets:
        parent_path = FIGURES_DIR / f"normality/MUV/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                arr = get_metrics(dataset, method, metric, get_test_splits(dataset))
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)
            if save_figs:
                save_name = f"dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break

In [ ]:
# MUV classification enrichment factor distributions 
if runner:
    logger.info("Plotting MUV classification enrichment factor distributions")
    metrics = ["EF10", "EF5", "EF1", "EF05"]
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    sns.set_theme(
        rc={
            "figure.figsize": (int(15*IMG_SIZE_SCALE), int(27*IMG_SIZE_SCALE)),
            "axes.labelsize": int(16*IMG_SIZE_SCALE),
            "xtick.labelsize": int(14*IMG_SIZE_SCALE),
            "ytick.labelsize": int(14*IMG_SIZE_SCALE),
            "axes.titlesize":int(20*IMG_SIZE_SCALE)
        }
    )
    break_loop = False
    for dataset in muv_datasets:
        parent_path = FIGURES_DIR / f"normality/MUV/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                arr = get_metrics(dataset, method, metric, get_test_splits(dataset))
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)
            if save_figs:
                save_name = f"EF_dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break


In [ ]:
# Tox21 classification distributions AUROC AUCPR MCC
if runner:
    logger.info("Plotting Tox21 classification distributions")
    metrics = ["AUROC", "AUCPR", "MCC"]
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    sns.set_theme(
        rc={
            "figure.figsize": (int(15*IMG_SIZE_SCALE), int(20*IMG_SIZE_SCALE)),
            "axes.labelsize": int(16*IMG_SIZE_SCALE),
            "xtick.labelsize": int(14*IMG_SIZE_SCALE),
            "ytick.labelsize": int(14*IMG_SIZE_SCALE),
            "axes.titlesize":int(20*IMG_SIZE_SCALE)
        }
    )
    break_loop = False
    for dataset in tox21_datasets:
        parent_path = FIGURES_DIR / f"normality/tox21/{dataset}"
        parent_path.mkdir(exist_ok=True, parents=True)
        for method in methods:
            method = get_method_to_plot(dataset, method)
            fig, grid = prep_grid(nrow=len(metrics))
            for i, metric in enumerate(metrics):
                arr = get_metrics(dataset, method, metric, get_test_splits(dataset))
                plot_distribution(arr, fig, grid, label=metric_display_names[metric], row=i)
            if save_figs:
                save_name = f"dist_{dataset}_{get_method_name(method, "s")}"
                plt.savefig(
                    parent_path / save_name,
                    bbox_inches="tight",
                )
            else: 
                plt.show()
                break_loop = True
            plt.close()
            if break_loop: break
        if break_loop: break

In [ ]:
if runner: IMG_SIZE_SCALE = default_img_size_scale

## Statistical Tests and Effect Sizes

In [ ]:
# rank-biserial correlation for effect size calculation of paired samples
def rank_biserial_correlation(arr_1, arr_2):
    """
    Calculate the rank-biserial correlation coefficients.

    Based on:
        Kerby, D. S. (2014). The simple difference formula: An approach to teaching nonparametric correlation.;
        Cureton, E. E. (1956). Rank-biserial correlation.
        Pingouin implementation: https://pingouin-stats.org/generated/pingouin.wilcoxon.html
    """
    diff = arr_1 - arr_2
    ranks = stats.rankdata(np.abs(diff))
    r_sum = np.sum((diff != 0) * ranks)
    if r_sum == 0:
        return 0.0
    r_plus = np.sum((diff > 0) * ranks)
    rbc = (2 * r_plus) / r_sum - 1
    return rbc


In [ ]:
# Effect size calculation
def calculate_effect_size(array_a, array_b, parametric=True):
    if parametric:
        effect = compute_effsize(array_a, array_b, eftype="cohen")
    else:
        # effect = cliffs_delta(array_a, array_b)[0]
        effect = rank_biserial_correlation(array_a, array_b)
    return effect

In [ ]:
# calculating effect sizes between arrays
# returns dataframe of effect sizes
def calculate_pairwise_effect(arrays, parametric=True, names=None):
    out = np.zeros((len(arrays), len(arrays)))
    for i,j in combinations(range(len(arrays)), 2):
        effect = calculate_effect_size(arrays[i], arrays[j], parametric=parametric)
        out[i, j] = effect
        out[j, i] = -effect
    out = pd.DataFrame(out, columns=names, index=names)
    return out

In [ ]:
# statistical tests for multiple comparisons 
# returns p-values
def multi_comparisons_test(arrays, parametric=True, names=None, p_adjust="bonferroni", **kwargs):
    if parametric:
        out = posthoc_tukey_hsd(arrays, p_adjust=p_adjust, **kwargs)
    else:
        # out = posthoc_dunn(arrays, p_adjust=p_adjust, **kwargs)
        out = posthoc_wilcoxon(arrays, p_adjust=p_adjust, **kwargs)
    if names:
        out = pd.DataFrame(out.to_numpy(), columns=names, index=names)
    return out

In [ ]:
# turn pvalues into discrete interger representation
def discrete_pvalues(pvalues, boundaries=(0.05, 0.01, 0.001, 0.0001)):
    pvalues_discrete = np.zeros_like(pvalues)
    for bound in boundaries:
        pvalues_discrete[pvalues < bound] += 1
    return pvalues_discrete

In [ ]:
# convert pvalue in * notation
def get_star(p):
    if p < 0.0001: return '****'
    elif p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return 'ns'

## MCSim Plots

In [ ]:
runner = run["sim_plot"]
if runner:
    logger.info("Running similarity distribution plots")

### Heatmap Functions

In [ ]:
def reorient(data, orientation="upper left", array=False):
    if isinstance(data, np.ndarray): 
        data = pd.DataFrame(data)
        array = True
    if orientation == "upper right":
        data = data
    elif orientation == "upper left":
        data = data[data.columns[::-1]]
        data = data.T
    elif orientation == "lower left":
        data = data.T
    elif orientation == "lower right":
        data = data.reindex(data.index[::-1])
    else:
        raise ValueError("Orientation must be one of `upper right`, `lower right`, `upper left`, `lower left`")

    if array:
        data = data.to_numpy()
    return data

In [ ]:
# plotting multiple comparisons similarity heatmap
def mcsim_plot(
    arrays, names, ax,
    parametric=True, vmax=1.0,
    pvalue_boundaries=(0.05, 0.01, 0.001, 0.0001),
    effect_cbar_ax=None, pval_cbar_ax=None,
    cbar_kws={}, annot_kws={"size": int(12*IMG_SIZE_SCALE)},
    orientation="upper left", cmap="vlag",
    xticks="top", yticks="left"
):
    effect_sizes = calculate_pairwise_effect(
        arrays, parametric=parametric, names=names
    )
    
    effect_sizes = reorient(effect_sizes, orientation)
    cbar_label_size = cbar_kws.pop("cbar_label_fontsize", int(18*IMG_SIZE_SCALE))
    cbar_tick_size = cbar_kws.pop("cbar_tick_fontsize", int(16*IMG_SIZE_SCALE))
    tick_length = cbar_kws.pop("tick_length", 10*IMG_SIZE_SCALE)
    tick_width = cbar_kws.pop("tick_width", 2*IMG_SIZE_SCALE)

    if effect_cbar_ax:
        effect_cbar = True
        cbar_kws["label"] = "Cohen's d" if parametric else "Rank-biserial correlation ($\\mathrm{r_{rb}}$) (Upper Triangle)" # "Cliff's $\\delta$" - older version

        effect_cbar_ax.tick_params(labelsize=cbar_tick_size, length=tick_length, width=tick_width)
        effect_cbar_ax.yaxis.label.set_size(cbar_label_size)
        effect_cbar_ax.xaxis.label.set_size(cbar_label_size)
        cbar_annot_fontsize = cbar_kws.pop("cbar_annot_fontsize", int(20*IMG_SIZE_SCALE))
        if cbar_kws.get("orientation", "vertical") == "horizontal":
            if "_r" in cmap:
                loc, ha = ((1.0, 1.2), (0.0, 1.2)), ("right", "left")
            else: 
                loc, ha = ((0.0, 1.2), (1.0, 1.2)), ("left", "right")
            effect_cbar_ax.text(
                *loc[0], "Favourable for x-axis", transform=effect_cbar_ax.transAxes,
                ha=ha[0], va='bottom', fontsize=cbar_annot_fontsize,
            )
            effect_cbar_ax.text(
                *loc[1], "Unfavourable for x-axis", transform=effect_cbar_ax.transAxes,
                ha=ha[1], va='bottom', fontsize=cbar_annot_fontsize,
            )
        else:
            if "_r" in cmap:
                loc, ha = ((0.0, 1.025), (0.0, -0.085)), ("center", "center")
            else:
                loc, ha = ((0.0, -0.085), (0.0, 1.025)), ("center", "center")
            effect_cbar_ax.text(
                *loc[0], "Favourable for x-axis", transform=effect_cbar_ax.transAxes,
                ha=ha[0], va='bottom', fontsize=cbar_annot_fontsize,
            )
            effect_cbar_ax.text(
                *loc[1], "Unfavourable for x-axis", transform=effect_cbar_ax.transAxes,
                ha=ha[1], va='bottom', fontsize=cbar_annot_fontsize,
            )

        
    else:
        effect_cbar = False

    mask = np.zeros_like(effect_sizes, dtype=bool)
    mask[np.tril_indices_from(mask)] = True
    mask[np.diag_indices_from(mask)] = True
    mask = reorient(mask, orientation)
    sns.heatmap(
        effect_sizes,
        mask=mask,
        annot=True,
        fmt=".3f",
        cmap=cmap,
        cbar=effect_cbar,
        vmax=vmax, vmin=-vmax,
        ax=ax,
        cbar_ax=effect_cbar_ax,
        cbar_kws=cbar_kws,
        annot_kws=annot_kws,
    )
    pvalues = multi_comparisons_test(
        arrays, parametric=parametric, names=names
    )
    pvalues = reorient(pvalues, orientation)

    mask = np.zeros_like(pvalues, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    mask[np.diag_indices_from(mask)] = True
    mask = reorient(mask, orientation)
    
    pval_annots = np.empty_like(pvalues).astype(str)
    for i in range(pvalues.shape[0]):
        for j in range(pvalues.shape[1]):
            if mask[i, j]: 
                pval_annots[i, j] = ""
            else:
                p = pvalues.iloc[i, j]
                pval_annots[i, j] = get_star(p)

    pvalues = discrete_pvalues(pvalues, pvalue_boundaries)
    
    colors = sns.color_palette("rocket_r", n_colors=(len(pvalue_boundaries)-1))
    # colors.insert(0, "#d3d3d3")
    colors.insert(0, "#ffffff")

    if effect_cbar:
        effect_cbar_ax.spines['outline'].set_linewidth(0.3) 
        effect_cbar_ax.spines['outline'].set_edgecolor('black')

    if pval_cbar_ax:
        pval_cbar = True
        cbar_label = "Tukey's HSD p-values" if parametric else "Wilcoxon signed-rank test p-values (Lower Triangle)" # "Dunn's Test p-values" - older version
        cbar_kws["label"] = cbar_label
    else:
        pval_cbar = False
                
    sns.heatmap(
        pvalues,
        mask=mask,
        cmap=colors,
        cbar=pval_cbar,
        ax=ax,
        vmin=-0.5, vmax=len(pvalue_boundaries)-0.5,
        cbar_ax=pval_cbar_ax,
        cbar_kws=cbar_kws,
        annot=pval_annots,
        annot_kws=annot_kws,
        fmt="",
    )
    if pval_cbar:
        pval_cbar_ax.tick_params(labelsize=cbar_tick_size, length=tick_length, width=tick_width)
        pval_cbar_ax.yaxis.label.set_size(cbar_label_size)
        pval_cbar_ax.xaxis.label.set_size(cbar_label_size)
        pval_cbar_ax.spines['outline'].set_linewidth(0.3) 
        pval_cbar_ax.spines['outline'].set_edgecolor('black')

        cbar_labels = [f"p-value > {pvalue_boundaries[0]}"]
        for i in range(1, len(pvalue_boundaries)):
            label = f"p-value < {pvalue_boundaries[i]}"
            cbar_labels.append(
                label
            )
        if cbar_kws.get("orientation", "vertical") == "horizontal":
            pval_cbar_ax.set_xticks(
                ticks=[i for i in range(len(pvalue_boundaries))], 
                labels=cbar_labels,
                fontsize=cbar_tick_size
            )
        else:
            pval_cbar_ax.set_yticks(
                ticks=[i for i in range(len(pvalue_boundaries))], 
                labels=cbar_labels,
                fontsize=cbar_tick_size
            )

    if yticks == "right":
        ax.yaxis.tick_right()
    if xticks == "top":
        ax.xaxis.tick_top()
    
    # Add diagonal line (bottom-left → top-right)
    n = pvalues.shape[0]
    ax.plot([0, n], [n, 0], color='black', linewidth=0.8)

    ax.tick_params(axis='y', which='both', length=0, pad=2,)
    ax.tick_params(axis='x', which='both', length=0, pad=7,)
    ax.set_xticklabels(effect_sizes.columns, rotation=0, va='top', ha='center')
    ax.set_yticklabels(effect_sizes.index, rotation=0, va='center', ha='right')

    return ax



### Regression

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (int(20*IMG_SIZE_SCALE), int(20*IMG_SIZE_SCALE)),
        "axes.labelsize": int(20*IMG_SIZE_SCALE),
        "xtick.labelsize": 16.5*IMG_SIZE_SCALE,
        "ytick.labelsize": 16.5*IMG_SIZE_SCALE,
        "axes.titlesize":int(20*IMG_SIZE_SCALE),
        "figure.subplot.hspace": 0.4,
        "figure.subplot.wspace": 0.3,
    }
)
annot_kws={"size": int(18*IMG_SIZE_SCALE)}

In [ ]:
# plot biogen effect sizes and pvalues
if runner:
    logger.info("Plotting Biogen effect sizes and p-values")
    methods = ["pt_gin", "scratch_gin", "ecfp", "sns", "fcfp"]
    metrics = ["R2", "Pearson", "MAPE"]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(3, 2,)
        axes = axes.ravel()
        cbar_height, cbar_width = 0.01, 0.8
        cbar_x, cbar_y, cbar_gap = 0.1, 0.97+(1-IMG_SIZE_SCALE)*0.05, 0.07
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal",
        }
        if metric == "MAPE": cmap = "vlag"
        else: cmap = "vlag_r"

        for i, (ax, dataset) in enumerate(zip(axes, biogen_datasets)):
            methods_to_plot = [get_method_to_plot(dataset, m) for m in methods]
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            method_names = [get_method_name(m) for m in methods_to_plot]
            ax = mcsim_plot(
                arrays=arrays, 
                names=method_names, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                annot_kws=annot_kws,
                cmap=cmap
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)
        if save_figs:
            plt.savefig(
                FIGURES_DIR / "biogen" / f"biogen_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop=True
        plt.close()
        if break_loop: break

In [ ]:
# plot chembl effect sizes and pvalues
if runner:
    logger.info("Plotting ChEMBL effect sizes and p-values")
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(1,2, figsize= (18*IMG_SIZE_SCALE,int(7*IMG_SIZE_SCALE)))
        axes = axes.ravel()
        cbar_height, cbar_width = 0.025, 0.8
        cbar_x, cbar_y, cbar_gap = 0.1, (1.2+(1-IMG_SIZE_SCALE)*0.025), (0.175+(1-IMG_SIZE_SCALE)*0.04)
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal",
        }
        if metric == "MAPE": cmap = "vlag"
        else: cmap = "vlag_r"

        for i, (ax, dataset) in enumerate(zip(axes, chembl_datasets)):
            methods_to_plot = [get_method_to_plot(dataset, m) for m in methods]
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            
            method_names = [get_method_name(m) for m in methods_to_plot]
            ax = mcsim_plot(
                arrays=arrays, 
                names=method_names, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                annot_kws=annot_kws,
                cmap=cmap
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)

        if save_figs:
            plt.savefig(
                FIGURES_DIR / "chembl" / f"chembl_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

In [ ]:
# plot molnet effect sizes and pvalues
if runner:
    logger.info("Plotting MoleculeNet regression effect sizes and p-values")
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(1,3, figsize= (27*IMG_SIZE_SCALE,int(7*IMG_SIZE_SCALE)))
        plt.subplots_adjust(wspace=0.35)
        axes = axes.ravel()
        cbar_height, cbar_width = 0.03, 0.75
        cbar_x, cbar_y, cbar_gap = 0.1, (1.2+(1-IMG_SIZE_SCALE)*0.025), (0.175+(1-IMG_SIZE_SCALE)*0.04)
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal"
        }
        if metric == "MAPE": cmap = "vlag"
        else: cmap = "vlag_r"

        for i, (ax, dataset) in enumerate(zip(axes, molnet_regression_datasets)):
            methods_to_plot = [get_method_to_plot(dataset, m) for m in methods]
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            method_names = [get_method_name(m) for m in methods_to_plot]
            ax = mcsim_plot(
                arrays=arrays, 
                names=method_names, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                annot_kws=annot_kws,
                cmap=cmap,
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)

        if save_figs:
            plt.savefig(
                FIGURES_DIR / "molnet" / f"molnet_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

In [ ]:
# plot tanimoto similarity
if runner:
    logger.info("Plotting Tanimoto similarity distributions")
    methods = [i/10 for i in range(5,11)]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(3, 2,)
        plt.subplots_adjust(wspace=0.15)
        axes = axes.ravel()
        cbar_height, cbar_width = 0.01, 0.8
        cbar_x, cbar_y, cbar_gap = 0.1, 1.0, 0.07
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal"
        }
        if metric == "MAPE": cmap = "vlag"
        else: cmap = "vlag_r"

        for i, (ax, dataset) in enumerate(zip(axes, biogen_datasets)):
            methods_to_plot = [
                get_method_to_plot(
                    dataset, m, key=lambda x, y: True if x.endswith(str(y).replace(".", "")) and "tanimoto" in x else False) 
                    for m in methods[1:]
                ]
            tanimoto_05 = methods_to_plot[0].split("_tanimoto")[0]
            methods_to_plot.insert(0, tanimoto_05)
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            effect = calculate_pairwise_effect(
                arrays, parametric=False, names=methods_to_plot
            )
            ax = mcsim_plot(
                arrays=arrays, 
                names=methods, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                annot_kws=annot_kws,
                cmap = cmap
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)
            
        if save_figs:
            plt.savefig(
                FIGURES_DIR / "biogen" / f"tanimoto_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

### Classification

In [ ]:
# plot muv effect sizes and pvalues
if runner:
    logger.info("Plotting MUV classification effect sizes and p-values")
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    metrics = ["AUROC", "AUCPR", "MCC", "EF10", "EF5", "EF1", "EF05"]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(5,4, figsize= (30*IMG_SIZE_SCALE,30*IMG_SIZE_SCALE),)
        plt.subplots_adjust(hspace=0.5, wspace=0.45)
        axes = axes.ravel()
        for i in range(1,4): axes[-i].set_visible(False)
        cbar_height, cbar_width = 0.01, 0.75
        cbar_x, cbar_y, cbar_gap = 0.1, (0.95 + (1-IMG_SIZE_SCALE)*0.025), (0.05 + (1-IMG_SIZE_SCALE)*0.01)
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal"
        }
        cmap = "vlag_r"
        for i, (ax, dataset) in enumerate(zip(axes, muv_datasets)):
            methods_to_plot = [get_method_to_plot(dataset, m) for m in methods]
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            method_names = [get_method_name(m) for m in methods_to_plot]
            ax = mcsim_plot(
                arrays=arrays, 
                names=method_names, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                annot_kws=annot_kws,
                cmap=cmap,
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)

        if save_figs:
            plt.savefig(
                FIGURES_DIR / "muv" / f"muv_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

In [ ]:
# plot tox21 effect sizes and pvalues
if runner:
    logger.info("Plotting Tox21 classification effect sizes and p-values")
    methods = ["pt_gin", "ecfp", "sns", "fcfp"]
    metrics = ["AUROC", "AUCPR", "MCC"]
    break_loop = False
    for metric in metrics:
        fig, axes = plt.subplots(4,3, figsize=(22*IMG_SIZE_SCALE,int(18*IMG_SIZE_SCALE)))
        plt.subplots_adjust(hspace=0.75, wspace=0.4)
        axes = axes.ravel()
        cbar_height, cbar_width = 0.01, 0.8
        cbar_x, cbar_y, cbar_gap = 0.1, (1.0 + (1-IMG_SIZE_SCALE)*0.04), (0.07 + (1-IMG_SIZE_SCALE)*0.01)
        pval_cbar_ax = fig.add_axes([cbar_x, cbar_y, cbar_width, cbar_height],)
        effect_cbar_ax = fig.add_axes([cbar_x, cbar_y+cbar_gap, cbar_width, cbar_height])
        cbar_kws={
            "orientation": "horizontal"
        }
        cmap = "vlag_r"

        for i, (ax, dataset) in enumerate(zip(axes, tox21_datasets)):
            methods_to_plot = [get_method_to_plot(dataset, m) for m in methods]
            arrays = [
                get_metrics(dataset, m, metric, get_test_splits(dataset)) 
                for m in methods_to_plot
            ]
            method_names = [get_method_name(m) for m in methods_to_plot]
            ax = mcsim_plot(
                arrays=arrays, 
                names=method_names, 
                parametric=False,
                ax=ax, 
                effect_cbar_ax=effect_cbar_ax if i==0 else None,
                pval_cbar_ax=pval_cbar_ax if i==0 else None,
                cbar_kws=cbar_kws,
                cmap=cmap
            )
            ax.set_title(f"({ascii_lowercase[i]}) {benchmark_dataset_names[dataset]}", loc="left", pad=int(20*IMG_SIZE_SCALE))
            render_box(fig, [ax],)

        if save_figs:
            plt.savefig(
                FIGURES_DIR / "tox21" / f"tox21_heatmap_{metric}",
                bbox_inches="tight",
            )
        else: 
            plt.show()
            break_loop = True
        plt.close()
        if break_loop: break

## Violin Plots

In [ ]:
runner = run["violin_plot"]
if runner:
    logger.info("Running violin plots")

### Violin Plot Functions

In [ ]:
# prepare dataframes for plotting violins with significances
def prepare_violin_df(
    datasets, metrics, methods, find_key=None, 
    parametric=True, tanimoto=False,
):
    out = {
        "Dataset": [],
        "Method": [],
    }
    pvalues_dict = {}
    for dataset in datasets:
        dataset_name = benchmark_dataset_names[dataset]
        if dataset == "ESOL_restricted": dset = "ESOL"
        else: dset = dataset
        methods_to_plot = [
            get_method_to_plot(dset, m, key=find_key) 
            for m in methods
        ]
        test_splits_idx = get_test_splits(dset)
        num_to_add = len(test_splits_idx) * len(methods_to_plot)
        out["Dataset"].extend([dataset_name] * num_to_add)
        add_method = True
        for metric in metrics:
            metric_name = metric_display_names[metric]
            if metric_name not in out:
                out[metric_name] = []
            arrays = []
            method_names_list = []
            for method in methods_to_plot:
                method_name = get_method_name(method, tanimoto=tanimoto)
                method_names_list.append(method_name)
                if add_method:
                    out["Method"].extend([method_name] * len(test_splits_idx))
                    
                arr = get_metrics(dataset, method, metric, test_splits_idx)
                out[metric_name].extend(arr)
                arrays.append(arr)
            pvalues = multi_comparisons_test(
                arrays=arrays, names=method_names_list, parametric=parametric,
            )
            pval_dataset_dict = pvalues_dict.get(metric, {})
            pval_dataset_dict[dataset_name] = pvalues
            pvalues_dict[metric] = pval_dataset_dict
            add_method = False

    return pd.DataFrame(out), pvalues_dict

In [ ]:
# get the best method by mean of a metric
def get_best_method(df, dataset, metric):
    df_subset = df[["Method", metric_display_names[metric]]]
    df_subset = df_subset[df.Dataset == dataset]
    df_subset = df_subset.groupby("Method").mean()
    if metric_sort_reverse[metric] == True:
        return df_subset.idxmax().iloc[0]
    else:
        return df_subset.idxmin().iloc[0]


In [ ]:
# check whether array contains only 1 unique value
# for MCC edge case with all 0s
def check_unique(df, dataset, metric, method):
    df_subset = df[metric_display_names[metric]][
        (df["Dataset"] == dataset) & (df["Method"] == method)
    ]
    return len(df_subset.unique()) > 1

In [ ]:
# add hatches to best method by mean
def add_hatches(df, metric, ax, hatch="///"):
    patches = [i for i in ax.get_children() if isinstance(i, PolyCollection)]
    count = 0
    best_methods_dict = {}
    for dataset, group in df.groupby("Dataset", sort=False):
        best_method = get_best_method(df, dataset, metric)
        best_methods_dict[dataset] = best_method
        for method in group["Method"].unique():
            if check_unique(df, dataset, metric, method):
                if method == best_method:
                    facecolor = patches[count].get_facecolor()[0]
                    rgb = facecolor[:3]
                    brightness = np.sqrt(0.299*rgb[0]**2 + 0.587*rgb[1]**2 + 0.114*rgb[2]**2)
                    if brightness < 0.5:
                        color = "lightgray"
                    else:
                        color = "black"
                    path = patches[count].get_paths()[0]
                    transform = patches[count].get_transform()

                    # Remove hatch from original
                    patches[count].set_hatch(None)

                    # Create a new patch on top with just the hatch
                    hatch_patch = PathPatch(
                        path,
                        transform=transform,
                        facecolor="none",
                        edgecolor=color,   # hatch color
                        hatch=hatch,
                        linewidth=0.0
                    )
                    ax.add_patch(hatch_patch)
                count += 1
    return ax, best_methods_dict


In [ ]:
# get the x axis position for each violin
def get_violin_pos(
    df, violin_width = 0.8,
):
    group_names = df['Dataset'].unique()
    hue_names = df['Method'].unique()
    n_hues = len(hue_names)

    # Get original tick positions (center of each group)
    group_positions = np.arange(len(group_names))

    # Get width of the total hue span within each group
    hue_offsets = np.linspace(
        violin_width*(-1+1/n_hues)/2, violin_width*(1-1/n_hues)/2,
        n_hues
    )

    # Build mapping from (group, hue) to x-position
    violin_pos = {
        (group, hue): group_positions[list(group_names).index(group)] + hue_offsets[list(hue_names).index(hue)]
        for group in group_names
        for hue in hue_names
    }
    return violin_pos

In [ ]:
# for each datasets get pvalue pairs between the best method and all other approaches
def get_pvalue_pairs(
    pvalues, best_methods,
):
    annots = []
    for dataset in pvalues:
        inner_annots = []
        pvalues_subset = pvalues[dataset]
        columns = pvalues_subset.columns
        best_method = best_methods[dataset]
        best_method_loc = columns.get_loc(best_method)
        for method in pvalues_subset.columns:
            if method == best_method: continue
            distance = np.abs(
                best_method_loc - columns.get_loc(method))
            pval = pvalues_subset.loc[best_method, method]
            pval = float(pval)
            inner_annots.append((
                (dataset, best_method), 
                (dataset, method),
                pval,
                distance
            ))
        inner_annots = list(sorted(
            inner_annots, key=lambda x: x[-1]
        ))

        annots.extend(inner_annots)
    return annots


In [ ]:
# add line between violins for significance annotation
def add_sig_line(
    ax, x1, x2, y, 
    pvalue,
    line_height, text_pad,
    line_width=1.5,
    fontsize=12,
    max_multiplier=1.075,
):
        
    annot = get_star(pvalue)
    if annot == 'ns': padding = 0
    else: padding = text_pad

    ax.plot(
        [x1, x1, x2, x2], 
        [y, y+line_height, y+line_height, y], 
        lw=line_width, c='k'
    )
    text_obj = ax.text(
        (x1+x2)*.5,
        y+line_height+padding,
        annot,
        ha='center', va='bottom', color='k',
        fontsize=fontsize,
    )
    fig = ax.get_figure()
    renderer = fig.canvas.get_renderer()
    bbox_pixels = text_obj.get_window_extent(renderer=renderer)
    bbox_data = bbox_pixels.transformed(ax.transData.inverted())
    y_min, y_max = ax.get_ylim()
    if bbox_data.ymax > y_max*1.05:
        ax.set_ylim(top=bbox_data.ymax*max_multiplier)

    return ax

In [ ]:
# plot all significance annotation on an axis
def significance_annots(
    df, pvalues, best_methods, metric,
    ax, violin_width=0.8,
    max_pad=1.,
    line_height=0.01,
    line_gap=0.01,
    text_pad=0.01,
    line_width=2.0*IMG_SIZE_SCALE,
    log_scale=False,
    fontsize=18,
    max_multiplier=1.075,
):
    fontsize = fontsize*IMG_SIZE_SCALE
    violin_pos = get_violin_pos(df, violin_width=violin_width,)
    pvalue_pairs = get_pvalue_pairs(pvalues, best_methods)
    group_ymax = df.groupby('Dataset')[metric].max().to_dict()
    group_steps = {}
    for (
        (dset_1, method_1), (dset_2, method_2), pvalue, _
    ) in pvalue_pairs:
       
        step = group_steps.get(dset_1, 0)
        x1 = violin_pos[(dset_1, method_1)]
        x2 = violin_pos[(dset_2, method_2)]
        if log_scale:
            y_max = group_ymax[dset_1]*max_pad
            y = y_max * (1+line_gap) ** (step + 1)
            h = y*line_height
            padding = y*text_pad
            
        else:
            y_max = group_ymax[dset_1] + (max_pad - 1)
            y = y_max + line_gap * step
            h = line_height
            padding = text_pad


        add_sig_line(
            ax=ax, x1=x1, x2=x2, y=y, 
            pvalue=pvalue, 
            line_height=h,
            text_pad=padding,
            line_width=line_width,
            fontsize=fontsize,
            max_multiplier=max_multiplier,
        )
        group_steps[dset_1] = step + 1

In [ ]:
# plot a violin with significance values
def violin_plot(
    df, metric, ax, colors=None,
    pvalues=None,
    inner=None,
    violin_width=0.8,
    log_scale=False,
    sig_kwargs={},
    sep_line={"color":"grey", "lw":1*IMG_SIZE_SCALE, "linestyle": "--"},
    hatch="///",
    **kwargs
):
    metric_name = metric_display_names[metric]
    ax = sns.violinplot(
        df, x="Dataset", y=metric_name, hue="Method",
        ax=ax, palette=colors, density_norm="width",
        inner=inner,
        width=violin_width,
        log_scale=log_scale,
        **kwargs
    )
    ax, best_methods = add_hatches(df, metric, ax, hatch=hatch)
    if pvalues is not None:
        pvalues = pvalues[metric]
        significance_annots(
            df, pvalues, best_methods, metric_name, ax, 
            violin_width=violin_width,
            log_scale=log_scale,
            **sig_kwargs
        )
    if sep_line is not None:
        for i in range(len(df['Dataset'].unique()) - 1):
            ax.axvline(i + 0.5, **sep_line)
    return ax

In [ ]:
# set color palette
method_color_palette = {
    i: j for i, j in zip(
        method_display_names.values(),
        sns.diverging_palette(145, 300, s=60, n=5),
    )
}

In [ ]:
grid_kwargs = {
    'major' : {
        'which': 'major',
        'axis': 'y',
        'color': 'black',
        'linestyle': ':',
        'linewidth': 0.6,
        'alpha': 0.5
    },
    'minor' : {
        'which': 'minor',
        'axis': 'y',
        'color': 'gray',
        'linestyle': ':',
        'linewidth': 0.6,
        'alpha': 0.3
    }
}

### Regression

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (30*IMG_SIZE_SCALE, 30*IMG_SIZE_SCALE),
        "axes.labelsize": 26*IMG_SIZE_SCALE,
        "xtick.labelsize": 26*IMG_SIZE_SCALE,
        "ytick.labelsize": 26*IMG_SIZE_SCALE,
        "axes.titlesize": 30*IMG_SIZE_SCALE,
        "figure.subplot.hspace": 0.35,
        "figure.subplot.wspace": 0.3,
        "ytick.minor.size": (10*IMG_SIZE_SCALE),
        "ytick.major.size": 13*IMG_SIZE_SCALE,
        "ytick.color": "black",
        "xtick.minor.size": (10*IMG_SIZE_SCALE),
        "xtick.major.size": 13*IMG_SIZE_SCALE,
        "xtick.color": "black",
        "hatch.linewidth": 2.0*IMG_SIZE_SCALE,
        "axes.linewidth": 1.5*IMG_SIZE_SCALE,
    },
    style="ticks", 
)

#### Biogen

In [ ]:
# get df and pvalues for biogen
if runner:
    logger.info("Preparing Biogen violin plot data")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin", "scratch_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(biogen_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "R2": {
        "max_pad": 3.37*IMG_SIZE_SCALE,
        "line_height": 0.0333*IMG_SIZE_SCALE,
        "line_gap": 0.2*IMG_SIZE_SCALE,
        "text_pad": -0.05*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "Pearson": {
        "max_pad": 3.37*IMG_SIZE_SCALE,
        "line_height": 0.0233*IMG_SIZE_SCALE,
        "line_gap": 0.14*IMG_SIZE_SCALE,
        "text_pad": -0.05*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "MAPE": {
        "max_pad": 3.0*IMG_SIZE_SCALE,
        "line_height": 0.1*IMG_SIZE_SCALE,
        "line_gap": 0.9*IMG_SIZE_SCALE,
        "text_pad": -0.3*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
        "max_multiplier": 2,
    }
}

In [ ]:
# run biogen plotting
if runner:
    logger.info("Plotting Biogen violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            hatch="/////",
            cut=0,
            linewidth=2.0*IMG_SIZE_SCALE,
        )
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        # if log_scale:
        #     ax.set_ylim(bottom=1e1)
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": 30*IMG_SIZE_SCALE},
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20*IMG_SIZE_SCALE)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "biogen" / "biogen_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

#### ChEMBL

In [ ]:
# get df and pvalues for biogen
if runner:
    logger.info("Preparing ChEMBL violin plot data")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(chembl_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "R2": {
        "max_pad": 3.36*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.1*IMG_SIZE_SCALE,
        "text_pad": -0.02*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
        "max_multiplier": 1.01,
    },
    "Pearson": {
        "max_pad": 3.36*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.0667*IMG_SIZE_SCALE,
        "text_pad": -0.01*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
        "max_multiplier": 1.01,
    },
    "MAPE": {
        "max_pad": 4.16*IMG_SIZE_SCALE,
        "line_height": 0.5*IMG_SIZE_SCALE,
        "line_gap": 2.0*IMG_SIZE_SCALE,
        "text_pad": -0.5*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
        "max_multiplier": 1.01,
    }
}

In [ ]:
# run chembl plotting
if runner:
    logger.info("Plotting ChEMBL violin plots")
    fig, axes = plt.subplots(3,1,)
    plt.subplots_adjust(hspace=0.4)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        log_scale = False
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
            cut=0
        )
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        # if log_scale:
            # ax.set_ylim(1e1, 1e3)
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])
        ax.set_ylim(top=ax.get_ylim()[1]*sig_kwargs[metrics[i]].get("max_multiplier", 1.075))

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": (30*IMG_SIZE_SCALE)}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=(20*IMG_SIZE_SCALE), loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "chembl" / "chembl_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

#### Molnet Regression

In [ ]:
# get df and pvalues for molnet
if runner:
    logger.info("Preparing MoleculeNet regression violin plot data")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(molnet_regression_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "R2": {
        "max_pad": 3.37*IMG_SIZE_SCALE,
        "line_height": 0.05*IMG_SIZE_SCALE,
        "line_gap": 0.125*IMG_SIZE_SCALE,
        "text_pad": -0.04*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "Pearson": {
        "max_pad": 3.37*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.125*IMG_SIZE_SCALE,
        "text_pad": -0.02*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "MAPE": {
        "max_pad": 16.7*IMG_SIZE_SCALE,
        "line_height": 10.0*IMG_SIZE_SCALE,
        "line_gap": 50.0*IMG_SIZE_SCALE,
        "text_pad": -15*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    }
}

In [ ]:
# run molnet plotting
if runner:
    logger.info("Plotting MoleculeNet regression violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        log_scale = False
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
            cut=0
        )
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])

        ax.set_ylim(top=ax.get_ylim()[1]*sig_kwargs[metrics[i]].get("max_multiplier", 1.01))

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": (30*IMG_SIZE_SCALE)}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "molnet" / "molnet_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

#### ESOL

In [ ]:
# get df and pvalues for esol
if runner:
    logger.info("Preparing ESOL regression violin plot data")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(esol_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "R2": {
        "max_pad": 3.5*IMG_SIZE_SCALE,
        "line_height": 0.125*IMG_SIZE_SCALE,
        "line_gap": 1.0*IMG_SIZE_SCALE,
        "text_pad": -0.3*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "Pearson": {
        "max_pad": 3.375*IMG_SIZE_SCALE,
        "line_height": 0.04*IMG_SIZE_SCALE,
        "line_gap": 0.13*IMG_SIZE_SCALE,
        "text_pad": -0.06*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    },
    "MAPE": {
        "max_pad": 12*IMG_SIZE_SCALE,
        "line_height": 10.0*IMG_SIZE_SCALE,
        "line_gap": 45*IMG_SIZE_SCALE,
        "text_pad": -10*IMG_SIZE_SCALE,
        "line_width": 2*IMG_SIZE_SCALE,
        "fontsize": 90*IMG_SIZE_SCALE,
    }
}

In [ ]:
# run esol plotting
if runner:
    logger.info("Plotting ESOL regression violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        log_scale = False
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            cut=0,
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
        )
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        if log_scale:
            ax.set_ylim(1e1, 1e3)

        ax.set_ylim(top=ax.get_ylim()[1]*sig_kwargs[metrics[i]].get("max_multiplier", 1.01))

        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": 30*IMG_SIZE_SCALE}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "molnet" / "esol_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

### Classification

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (30*IMG_SIZE_SCALE, 30*IMG_SIZE_SCALE),
        "axes.labelsize": 22*IMG_SIZE_SCALE,
        "xtick.labelsize": 20*IMG_SIZE_SCALE,
        "ytick.labelsize": 20*IMG_SIZE_SCALE,
        "axes.titlesize":24*IMG_SIZE_SCALE,
        "figure.subplot.hspace": 0.3,
        "figure.subplot.wspace": 0.3,
        "ytick.minor.size": (18*IMG_SIZE_SCALE),
        "ytick.major.size": 18*IMG_SIZE_SCALE,
        "ytick.color": "black",
        "xtick.minor.size": (18*IMG_SIZE_SCALE),
        "xtick.major.size": 18*IMG_SIZE_SCALE,
        "xtick.color": "black",
        "hatch.linewidth": 2.0*IMG_SIZE_SCALE,
        "axes.linewidth": 1.5*IMG_SIZE_SCALE,
    },
    style="ticks",
)

#### MUV

##### AUROC, AUCPR, MCC

In [ ]:
# get df and pvalues for muv
if runner:
    logger.info("Preparing MUV classification violin plot data")
    metrics = ["AUROC", "AUCPR", "MCC",]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(muv_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "AUROC": {
        "max_pad": 3.36*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.165*IMG_SIZE_SCALE,
        "text_pad": -0.045*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    },
    "AUCPR": {
        "max_pad": 3.36*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.155*IMG_SIZE_SCALE,
        "text_pad": -0.045*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE,
    },
    "MCC": {
        "max_pad": 3.36*IMG_SIZE_SCALE,
        "line_height": 0.0167*IMG_SIZE_SCALE,
        "line_gap": 0.135*IMG_SIZE_SCALE,
        "text_pad": -0.045*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE,
    }
}

In [ ]:
# run muv metric plotting
if runner:
    logger.info("Plotting MUV classification violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = False
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            cut=0,
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
        )

        # Filter out ticks outside the range
        yticks = ax.get_yticks()
        ymin, ymax = metric_range_limits[metrics[i]]
        valid_yticks = [y for y in yticks if ymin <= round(y, 5) <= ymax]
        ax.set_yticks(valid_yticks)
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  

        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])
        ax.set_ylim(top=ax.get_ylim()[1]*sig_kwargs[metrics[i]].get("max_multiplier", 1.01))

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": 30*IMG_SIZE_SCALE}
            )
        ax.set_xlabel("")
        
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "muv" / "muv_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

##### EF

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (35*IMG_SIZE_SCALE, 30*IMG_SIZE_SCALE),
        "axes.labelsize": 24*IMG_SIZE_SCALE,
        "xtick.labelsize": 22*IMG_SIZE_SCALE,
        "ytick.labelsize": 24*IMG_SIZE_SCALE,
        "axes.titlesize":24*IMG_SIZE_SCALE,
        "figure.subplot.hspace": 0.45,
        "figure.subplot.wspace": 0.3,
        "ytick.minor.size": (20*IMG_SIZE_SCALE),
        "ytick.major.size": 20*IMG_SIZE_SCALE,
        "ytick.color": "black",
        "xtick.minor.size": (20*IMG_SIZE_SCALE),
        "xtick.major.size": 20*IMG_SIZE_SCALE,
        "xtick.color": "black",
        "hatch.linewidth": 2.0*IMG_SIZE_SCALE,
        "axes.linewidth": 1.5*IMG_SIZE_SCALE,
    },
    style="ticks",
)

In [ ]:
# get df and pvalues for muv
if runner:
    logger.info("Preparing MUV Enrichment violin plot data")
    metrics = ["EF10", "EF5", "EF1", "EF05"]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(muv_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "EF10": {
        "max_pad": 1.25,
        "line_height": 0.15,
        "line_gap": 1.1,
        "text_pad": -1*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE,
    },
    "EF5": {
        "max_pad": 1.5,
        "line_height": 0.25,
        "line_gap": 2.3,
        "text_pad": -2.5*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE,
    },
    "EF1": {
        "max_pad": 3.0,
        "line_height": 1.0,
        "line_gap": 10.0,
        "text_pad": -10*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE,
    },
    "EF05": {
        "max_pad": 6.0,
        "line_height": 2,
        "line_gap": 18.0,
        "text_pad": -20.0*IMG_SIZE_SCALE,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize":80*IMG_SIZE_SCALE
    }
}

In [ ]:
# run muv metric plotting
if runner:
    logger.info("Plotting MUV EF violin plots")
    fig, axes = plt.subplots(4,1,)
    axes = axes.ravel()
    for i in range(len(axes)):

        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            cut=0,
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
        )

        # Filter out ticks outside the range
        yticks = ax.get_yticks()
        ymin, ymax = metric_range_limits[metrics[i]]
        valid_yticks = [y for y in yticks if ymin <= round(y, 5) <= ymax]
        ax.set_yticks(valid_yticks)
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        if metrics[i] == "EF10":
            ax.set_ylim(ymin, ymax*1.35)
        elif metrics[i] == "EF5":
            ax.set_ylim(ymin, ymax*1.32)
        elif metrics[i] == "EF1":
            ax.set_ylim(ymin, ymax*1.11)
        else:
            ax.set_ylim(ymin, ymax*1.05)
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": 30*IMG_SIZE_SCALE}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "muv" / "muv_ef_violins.pdf",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

#### Tox21

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (30*IMG_SIZE_SCALE, 30*IMG_SIZE_SCALE),
        "axes.labelsize": 24*IMG_SIZE_SCALE,
        "xtick.labelsize": 22*IMG_SIZE_SCALE,
        "ytick.labelsize": 22*IMG_SIZE_SCALE,
        "axes.titlesize":24*IMG_SIZE_SCALE,
        "figure.subplot.hspace": 0.4,
        "figure.subplot.wspace": 0.3,
        "ytick.minor.size": (18*IMG_SIZE_SCALE),
        "ytick.major.size": 18*IMG_SIZE_SCALE,
        "ytick.color": "black",
        "xtick.minor.size": (18*IMG_SIZE_SCALE),
        "xtick.major.size": 18*IMG_SIZE_SCALE,
        "xtick.color": "black",
        "hatch.linewidth": 2.0*IMG_SIZE_SCALE,
        "axes.linewidth": 1.5*IMG_SIZE_SCALE,
    },
    style="ticks",
)


In [ ]:
# get df and pvalues for tox21
if runner:
    logger.info("Preparing Tox21 classification violin plot data")
    metrics = ["AUROC", "AUCPR", "MCC",]
    methods = ["pt_gin", "sns", "ecfp", "fcfp"]
    df, pvalues_dict = prepare_violin_df(tox21_datasets, metrics, methods, parametric=False,)

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "AUROC": {
        "max_pad": 1.005,
        "line_height": 0.0025,
        "line_gap": 0.022,
        "text_pad": -0.0075,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    },
    "AUCPR": {
        "max_pad": 1.01,
        "line_height": 0.01,
        "line_gap": 0.055,
        "text_pad": -0.015,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    },
    "MCC": {
        "max_pad": 1.01,
        "line_height": 0.01,
        "line_gap": 0.055,
        "text_pad": -0.015,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    }
}

In [ ]:
# run tox21 metric plotting
if runner:
    logger.info("Plotting Tox21 classification violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            cut=0,
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
        )

        # Filter out ticks outside the range
        yticks = ax.get_yticks()
        ymin, ymax = metric_range_limits[metrics[i]]
        valid_yticks = [y for y in yticks if ymin <= round(y, 5) <= ymax]
        ax.set_yticks(valid_yticks)
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black')  
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=6,
                prop={"size": 30*IMG_SIZE_SCALE}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "tox21" / "tox_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()

### Tanimoto Similarity 

In [ ]:
# set sns theme
sns.set_theme(
    rc={
        "figure.figsize": (25*IMG_SIZE_SCALE, 30*IMG_SIZE_SCALE),
        "axes.labelsize": 24*IMG_SIZE_SCALE,
        "xtick.labelsize": 22*IMG_SIZE_SCALE,
        "ytick.labelsize": 22*IMG_SIZE_SCALE,
        "axes.titlesize":24*IMG_SIZE_SCALE,
        "figure.subplot.hspace": 0.3,
        "figure.subplot.wspace": 0.3,
        "ytick.minor.size": (18*IMG_SIZE_SCALE),
        "ytick.major.size": 18*IMG_SIZE_SCALE,
        "ytick.color": "black",
        "xtick.minor.size": (18*IMG_SIZE_SCALE),
        "xtick.major.size": 18*IMG_SIZE_SCALE,
        "xtick.color": "black",
        "hatch.linewidth": 2.0*IMG_SIZE_SCALE,
        "axes.linewidth": 1.5*IMG_SIZE_SCALE,
    },
    style="ticks",
)

In [ ]:
# set color palette
method_color_palette = {
    str(i/10): j for i, j in zip(
        range(5,11),
        sns.color_palette("Blues", n_colors=6),
    )
}
method_color_palette["05"] = (1.,1.,1.)

In [ ]:
# get df and pvalues for biogen
if runner:
    logger.info("Preparing Biogen Tanimoto violin plot data")
    metrics = ["R2", "Pearson", "MAPE"]
    methods = ["pt_gin",] + [str(i/10).replace(".", "") for i in range(6,11)]
    df, pvalues_dict = prepare_violin_df(
        biogen_datasets, metrics, methods, parametric=False, 
        find_key = lambda x, y: True 
        if (y in methods[1:]) and (x[-2:] == y) and ("tanimoto" in x) 
        else True if (y == "pt_gin") and (y in x) and ("tanimoto_filter" not in x)
        else False,
        tanimoto=True
    )

In [ ]:
# set significance annotation kwargs
sig_kwargs = {
    "R2": {
        "max_pad": 1.005,
        "line_height": 0.0075,
        "line_gap": 0.0395,
        "text_pad": -0.0125,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    },
    "Pearson": {
        "max_pad": 1.005,
        "line_height": 0.005,
        "line_gap": 0.0285,
        "text_pad": -0.01,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
    },
    "MAPE": {
        "max_pad": 0.85,
        "line_height": 0.05,
        "line_gap": 0.35,
        "text_pad": -0.1,
        "line_width": 2.0*IMG_SIZE_SCALE,
        "fontsize": 80*IMG_SIZE_SCALE,
        "max_multiplier":1.25
    }
}

In [ ]:
# run biogen plotting
if runner:
    logger.info("Plotting Biogen Tanimoto violin plots")
    fig, axes = plt.subplots(3,1,)
    axes = axes.ravel()
    for i in range(len(axes)):
        log_scale = metrics[i] == "MAPE"
        ax = violin_plot(
            df=df, metric=metrics[i], ax=axes[i], 
            colors=method_color_palette,
            pvalues=pvalues_dict,
            log_scale=log_scale,
            sig_kwargs=sig_kwargs[metrics[i]],
            cut=0,
            hatch="/////",
            linewidth=2.0*IMG_SIZE_SCALE,
        )
        if log_scale:
            ax.set_ylim(1e1, 2e3)
        yticks = ax.get_yticks()
        ymin, ymax = metric_range_limits[metrics[i]]
        valid_yticks = [y for y in yticks if ymin <= round(y, 5) <= ymax]
        ax.tick_params(axis='y', which='minor', length=5*IMG_SIZE_SCALE,)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE)
        ax.tick_params(axis='y', which='major', length=8*IMG_SIZE_SCALE, width=1.2)
        ax.tick_params(axis='y', which='minor', length=4*IMG_SIZE_SCALE, color='black') 
        
        ax.grid(True, **grid_kwargs['major'])
        ax.grid(True, **grid_kwargs['minor'])
        ax.set_ylim(top=ax.get_ylim()[1]*sig_kwargs[metrics[i]].get("max_multiplier", 1.01))

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            handles.append(
                Patch(
                    edgecolor="black",
                    facecolor="white",
                    hatch="////",
                    label="Best Mean",
                    linewidth=2.0*IMG_SIZE_SCALE,
                )
            )
            fig.legend(
                handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.925),
                ncol=7,
                prop={"size": 30*IMG_SIZE_SCALE}
            )
        ax.set_xlabel("")
        # ax.set_xlabel("Dataset", labelpad=20)
        ax.set_title(f"({ascii_lowercase[i]})", pad=20*IMG_SIZE_SCALE, loc="left")
        ax.get_legend().remove()
        render_box(fig, [ax],)
    if save_figs:
        plt.savefig(
            FIGURES_DIR / "biogen" / "tanimoto_violins",
            bbox_inches="tight",
        )
    else: 
        plt.show()
    plt.close()